# 1st Hidden Layer — Evaluate Little-Perturbation Checkpoints

In the 1st-layer perturbation sweeps, models trained with a *little*
perturbation tend to score better at their matched evaluation level. This
notebook isolates that effect: for each perturbation type it loads a **single
checkpoint trained at a small perturbation level** and evaluates that one model
across the **entire** perturbation sweep (eval-on-checkpoint), rather than
training a fresh model per level.

Each perturbation is evaluated twice — once for the **no-delay** model and once
for the **delay** model — using the same checkpoint level:

- **Jitter** — per-spike Gaussian jitter; default checkpoint `sigma = 5`.
- **Shift** — per-neuron Gaussian shift; default checkpoint `sigma = 5`.
- **Deletion** — per-spike deletion; default checkpoint `p_d = 0.2`.

Evaluations cover the **whole / part / norm** SHD variants. Per-section results
are written to `log_eval_on_perturbatedModel/` with a `1stLayer` tag so they
stay distinct from the 2nd-layer eval-on-checkpoint outputs.

In [1]:
# Setup: reuse the 1st-layer training modules (model classes, data pipeline
# and evaluation routines) so the evaluation matches the original sweeps.
import sys
import json
from pathlib import Path

import numpy as np
import torch

BASE_DIR = Path.cwd()
assert (BASE_DIR / "jitter").is_dir(), (
    "Run this notebook from my_project/code/perturbation/ so the "
    "jitter / shift / deletion packages are importable."
)

for sub in ("jitter", "shift", "deletion"):
    sub_path = str((BASE_DIR / sub).resolve())
    if sub_path not in sys.path:
        sys.path.append(sub_path)

import jitter_train as jitter_mod
import shift_train as shift_mod
import deletion_train as deletion_mod

device = jitter_mod.device
print(f"Device: {device}")

# Dataset variants to evaluate (matches result_visualization_1stLayer.ipynb).
EVAL_DATASETS = ("whole", "part", "norm")

# Destination for the eval-on-checkpoint sweep results.
EVAL_LOG_DIR = Path("log_eval_on_perturbatedModel")
EVAL_LOG_DIR.mkdir(parents=True, exist_ok=True)
print(f"Eval results dir: {EVAL_LOG_DIR.resolve()}")

Using device: cuda
Using device: cuda
Using device: cuda
Device: cuda
Eval results dir: D:\IC_2025\IRP\workspace\my_project\code\perturbation\log_eval_on_perturbatedModel


## Shared evaluation helpers

`DATASET_CONFIGS`, `SIM_PARAMS` and the split logic are identical across the
three training modules, so the helpers below read them from `jitter_mod`.
`test_with_repeats` takes the perturbation level as its third positional
argument (`sigma` for jitter/shift, `p_d` for deletion), so one evaluation
routine serves all three perturbation types and both delay modes.

The only difference from the 2nd-layer notebook is the checkpoint filename:
1st-layer checkpoints carry no `_2ndLayer` infix
(`{perturbation}_{dataset}_{delay}_{token}.pt`), while the output JSON is
tagged with `1stLayer` so both layers can share `EVAL_LOG_DIR`.

In [2]:
_TEST_LOADER_CACHE: dict[str, object] = {}


def get_test_loader(dataset_key: str):
    """Return the test DataLoader for ``dataset_key``.

    Uses the same split ranges and seed as training, so the evaluation set
    matches the one behind the original sweep results. Cached because the
    test data is identical across perturbation types and delay modes.
    """
    if dataset_key not in _TEST_LOADER_CACHE:
        cfg = jitter_mod.DATASET_CONFIGS[dataset_key]
        X, Y = jitter_mod.load_shd_data(
            cfg["mat_file"], target_T=jitter_mod.SIM_PARAMS["tSample"]
        )
        _, _, test_loader = jitter_mod.build_dataloaders(
            X, Y, batch_size=jitter_mod.BATCH_SIZE, seed=jitter_mod.SEED
        )
        _TEST_LOADER_CACHE[dataset_key] = test_loader
    return _TEST_LOADER_CACHE[dataset_key]


def load_checkpoint(module, net_class, dataset_key, ckpt_filename, use_delay):
    """Instantiate a network (delay or no-delay) and load a saved checkpoint."""
    cfg = module.DATASET_CONFIGS[dataset_key]
    net = net_class(
        input_dim=cfg["input_dim"],
        hidden_units=module.HIDDEN_UNITS,
        num_classes=module.NUM_CLASSES,
        use_delay=use_delay,
        max_delay=module.MAX_DELAY,
    ).to(module.device)
    ckpt_path = module.DATA_DIR / ckpt_filename
    state = torch.load(ckpt_path, map_location=module.device)
    net.load_state_dict(state)
    net.eval()
    return net


def evaluate_across_levels(module, net, dataset_key, eval_levels):
    """Evaluate one fixed model at every perturbation level."""
    test_loader = get_test_loader(dataset_key)
    results = {}
    for level in eval_levels:
        res = module.test_with_repeats(net, test_loader, level)
        results[level] = res
        print(f"      eval@{level}: {res['mean']:.4f} +/- {res['std']:.4f}")
    return results


def save_sweep_json(results, out_path, key_fn):
    """Serialise eval results to the same schema as the training sweeps."""
    serial = {
        key_fn(level): {
            "mean": float(d["mean"]),
            "std": float(d["std"]),
            "values": [float(v) for v in d["values"]],
        }
        for level, d in results.items()
    }
    with open(out_path, "w") as fp:
        json.dump(serial, fp, indent=2)
    print(f"  saved -> {out_path}")


def run_eval(module, net_class, perturbation, delay_tag, ckpt_token,
             eval_levels, key_fn):
    """Evaluate the ``perturbation``/``delay_tag`` checkpoint across all levels.

    Loads ``{perturbation}_{ds}_{delay_tag}_{ckpt_token}.pt`` for every
    dataset, evaluates it at each level in ``eval_levels`` and writes one
    ``{perturbation}_1stLayer_{ds}_{delay_tag}_evalon_{ckpt_token}.json`` per
    dataset to ``EVAL_LOG_DIR``.
    """
    use_delay = delay_tag == "delay"
    for dataset_key in EVAL_DATASETS:
        ckpt_file = (
            f"{perturbation}_{dataset_key}_{delay_tag}_{ckpt_token}.pt"
        )
        print(f"[{perturbation}/{delay_tag}] dataset={dataset_key} "
              f"| checkpoint={ckpt_file}")
        net = load_checkpoint(
            module, net_class, dataset_key, ckpt_file, use_delay
        )
        results = evaluate_across_levels(module, net, dataset_key, eval_levels)
        out_path = EVAL_LOG_DIR / (
            f"{perturbation}_1stLayer_{dataset_key}_{delay_tag}_"
            f"evalon_{ckpt_token}.json"
        )
        save_sweep_json(results, out_path, key_fn=key_fn)

## 1a. Jitter (per-spike) — no delay

Evaluate the **no-delay** checkpoint trained at `CHECKPOINT_SIGMA_JITTER` across
the full jitter sweep (`jitter_mod.SIGMA_VALUES`).

In [3]:
# Jitter level of the checkpoint to evaluate (model trained at this sigma).
# Shared by the no-delay (1a) and delay (1b) sub-sections.
CHECKPOINT_SIGMA_JITTER = 5

run_eval(
    jitter_mod, jitter_mod.JitterSHDNetwork,
    perturbation="jitter", delay_tag="nodelay",
    ckpt_token=f"sigma{CHECKPOINT_SIGMA_JITTER}",
    eval_levels=jitter_mod.SIGMA_VALUES,
    key_fn=lambda lvl: str(int(lvl)),
)

[jitter/nodelay] dataset=whole | checkpoint=jitter_whole_nodelay_sigma5.pt


d:\IC_2025\IRP\workspace\venv\Lib\site-packages\torch\nn\utils\weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Padded time dimension from 100 to 200
Loaded D:\IC_2025\IRP\workspace\my_project\code\perturbation\jitter\..\..\realistic\shd\shd_data\shd_whole.mat: X=(9984, 700, 200), Y=(9984,), classes=20
Train: 5990 | Val: 1498 | Test: 1497
      eval@0: 0.3841 +/- 0.0000
      eval@1: 0.4309 +/- 0.0041
      eval@3: 0.6322 +/- 0.0030
      eval@5: 0.7043 +/- 0.0051
      eval@10: 0.6591 +/- 0.0114
      eval@17: 0.5498 +/- 0.0057
      eval@25: 0.4674 +/- 0.0044
  saved -> log_eval_on_perturbatedModel\jitter_1stLayer_whole_nodelay_evalon_sigma5.json
[jitter/nodelay] dataset=part | checkpoint=jitter_part_nodelay_sigma5.pt
Padded time dimension from 100 to 200
Loaded D:\IC_2025\IRP\workspace\my_project\code\perturbation\jitter\..\..\realistic\shd\shd_data\shd_part_new.mat: X=(5460, 224, 200), Y=(5460,), classes=20
Train: 3276 | Val: 819 | Test: 819
      eval@0: 0.2344 +/- 0.0000
      eval@1: 0.2755 +/- 0.0113
      eval@3: 0.4021 +/- 0.0058
      eval@5: 0.4717 +/- 0.0116
      eval@10: 0.4001 +/

## 1b. Jitter (per-spike) — delay

Same experiment on the **delay** model: evaluate the delay checkpoint trained at
`CHECKPOINT_SIGMA_JITTER` across the full jitter sweep.

In [4]:
run_eval(
    jitter_mod, jitter_mod.JitterSHDNetwork,
    perturbation="jitter", delay_tag="delay",
    ckpt_token=f"sigma{CHECKPOINT_SIGMA_JITTER}",
    eval_levels=jitter_mod.SIGMA_VALUES,
    key_fn=lambda lvl: str(int(lvl)),
)

[jitter/delay] dataset=whole | checkpoint=jitter_whole_delay_sigma5.pt
      eval@0: 0.6941 +/- 0.0000
      eval@1: 0.7326 +/- 0.0041
      eval@3: 0.8183 +/- 0.0068
      eval@5: 0.8290 +/- 0.0014
      eval@10: 0.7310 +/- 0.0067
      eval@17: 0.6199 +/- 0.0047
      eval@25: 0.5520 +/- 0.0014
  saved -> log_eval_on_perturbatedModel\jitter_1stLayer_whole_delay_evalon_sigma5.json
[jitter/delay] dataset=part | checkpoint=jitter_part_delay_sigma5.pt
      eval@0: 0.4164 +/- 0.0000
      eval@1: 0.4774 +/- 0.0026
      eval@3: 0.5963 +/- 0.0072
      eval@5: 0.6264 +/- 0.0036
      eval@10: 0.4363 +/- 0.0120
      eval@17: 0.2792 +/- 0.0083
      eval@25: 0.2243 +/- 0.0051
  saved -> log_eval_on_perturbatedModel\jitter_1stLayer_part_delay_evalon_sigma5.json
[jitter/delay] dataset=norm | checkpoint=jitter_norm_delay_sigma5.pt
      eval@0: 0.2149 +/- 0.0000
      eval@1: 0.2495 +/- 0.0042
      eval@3: 0.3516 +/- 0.0070
      eval@5: 0.3614 +/- 0.0069
      eval@10: 0.2214 +/- 0.0046
   

## 2a. Shift (per-neuron) — no delay

Evaluate the **no-delay** checkpoint trained at `CHECKPOINT_SIGMA_SHIFT` across
the full shift sweep (`shift_mod.SIGMA_VALUES`).

In [5]:
# Shift level of the checkpoint to evaluate (model trained at this sigma).
# Shared by the no-delay (2a) and delay (2b) sub-sections.
CHECKPOINT_SIGMA_SHIFT = 5

run_eval(
    shift_mod, shift_mod.ShiftSHDNetwork,
    perturbation="shift", delay_tag="nodelay",
    ckpt_token=f"sigma{CHECKPOINT_SIGMA_SHIFT}",
    eval_levels=shift_mod.SIGMA_VALUES,
    key_fn=lambda lvl: str(int(lvl)),
)

[shift/nodelay] dataset=whole | checkpoint=shift_whole_nodelay_sigma5.pt
      eval@0: 0.4115 +/- 0.0000
      eval@1: 0.4447 +/- 0.0057
      eval@3: 0.5952 +/- 0.0091
      eval@5: 0.6448 +/- 0.0103
      eval@10: 0.5536 +/- 0.0021
      eval@17: 0.3946 +/- 0.0096
      eval@25: 0.3031 +/- 0.0038
  saved -> log_eval_on_perturbatedModel\shift_1stLayer_whole_nodelay_evalon_sigma5.json
[shift/nodelay] dataset=part | checkpoint=shift_part_nodelay_sigma5.pt
      eval@0: 0.2418 +/- 0.0000
      eval@1: 0.2853 +/- 0.0072
      eval@3: 0.4103 +/- 0.0082
      eval@5: 0.4314 +/- 0.0063
      eval@10: 0.3276 +/- 0.0146
      eval@17: 0.1950 +/- 0.0068
      eval@25: 0.1282 +/- 0.0065
  saved -> log_eval_on_perturbatedModel\shift_1stLayer_part_nodelay_evalon_sigma5.json
[shift/nodelay] dataset=norm | checkpoint=shift_norm_nodelay_sigma5.pt
      eval@0: 0.1221 +/- 0.0000
      eval@1: 0.1571 +/- 0.0047
      eval@3: 0.2527 +/- 0.0112
      eval@5: 0.2979 +/- 0.0069
      eval@10: 0.1864 +/- 0.

## 2b. Shift (per-neuron) — delay

Same experiment on the **delay** model: evaluate the delay checkpoint trained at
`CHECKPOINT_SIGMA_SHIFT` across the full shift sweep.

In [6]:
run_eval(
    shift_mod, shift_mod.ShiftSHDNetwork,
    perturbation="shift", delay_tag="delay",
    ckpt_token=f"sigma{CHECKPOINT_SIGMA_SHIFT}",
    eval_levels=shift_mod.SIGMA_VALUES,
    key_fn=lambda lvl: str(int(lvl)),
)

[shift/delay] dataset=whole | checkpoint=shift_whole_delay_sigma5.pt
      eval@0: 0.7361 +/- 0.0000
      eval@1: 0.7646 +/- 0.0044
      eval@3: 0.8176 +/- 0.0009
      eval@5: 0.8054 +/- 0.0052
      eval@10: 0.6270 +/- 0.0089
      eval@17: 0.3946 +/- 0.0096
      eval@25: 0.3137 +/- 0.0083
  saved -> log_eval_on_perturbatedModel\shift_1stLayer_whole_delay_evalon_sigma5.json
[shift/delay] dataset=part | checkpoint=shift_part_delay_sigma5.pt
      eval@0: 0.5495 +/- 0.0000
      eval@1: 0.5527 +/- 0.0040
      eval@3: 0.6573 +/- 0.0055
      eval@5: 0.6361 +/- 0.0087
      eval@10: 0.3810 +/- 0.0121
      eval@17: 0.2214 +/- 0.0073
      eval@25: 0.1530 +/- 0.0055
  saved -> log_eval_on_perturbatedModel\shift_1stLayer_part_delay_evalon_sigma5.json
[shift/delay] dataset=norm | checkpoint=shift_norm_delay_sigma5.pt
      eval@0: 0.2430 +/- 0.0000
      eval@1: 0.2861 +/- 0.0057
      eval@3: 0.3492 +/- 0.0036
      eval@5: 0.3460 +/- 0.0049
      eval@10: 0.2002 +/- 0.0026
      eval@

## 3a. Deletion (per-spike) — no delay

Evaluate the **no-delay** checkpoint trained at `CHECKPOINT_PD_DELETION` across
the full deletion sweep (`deletion_mod.PD_VALUES`).

In [3]:
# Deletion probability of the checkpoint to evaluate (model trained at this p_d).
# Shared by the no-delay (3a) and delay (3b) sub-sections.
CHECKPOINT_PD_DELETION = 0.2
_pd_token = f"pd{int(round(CHECKPOINT_PD_DELETION * 10)):02d}"

run_eval(
    deletion_mod, deletion_mod.DeletionSHDNetwork,
    perturbation="deletion", delay_tag="nodelay",
    ckpt_token=_pd_token,
    eval_levels=deletion_mod.PD_VALUES,
    key_fn=lambda lvl: str(float(lvl)),
)

[deletion/nodelay] dataset=whole | checkpoint=deletion_whole_nodelay_pd02.pt


d:\IC_2025\IRP\workspace\venv\Lib\site-packages\torch\nn\utils\weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Padded time dimension from 100 to 200
Loaded D:\IC_2025\IRP\workspace\my_project\code\perturbation\jitter\..\..\realistic\shd\shd_data\shd_whole.mat: X=(9984, 700, 200), Y=(9984,), classes=20
Train: 5990 | Val: 1498 | Test: 1497
      eval@0.0: 0.5711 +/- 0.0000
      eval@0.2: 0.5446 +/- 0.0017
      eval@0.4: 0.4977 +/- 0.0104
      eval@0.6: 0.3990 +/- 0.0122
      eval@0.8: 0.2710 +/- 0.0019
  saved -> log_eval_on_perturbatedModel\deletion_1stLayer_whole_nodelay_evalon_pd02.json
[deletion/nodelay] dataset=part | checkpoint=deletion_part_nodelay_pd02.pt
Padded time dimension from 100 to 200
Loaded D:\IC_2025\IRP\workspace\my_project\code\perturbation\jitter\..\..\realistic\shd\shd_data\shd_part_new.mat: X=(5460, 224, 200), Y=(5460,), classes=20
Train: 3276 | Val: 819 | Test: 819
      eval@0.0: 0.4579 +/- 0.0000
      eval@0.2: 0.4013 +/- 0.0085
      eval@0.4: 0.3325 +/- 0.0089
      eval@0.6: 0.2336 +/- 0.0060
      eval@0.8: 0.1449 +/- 0.0106
  saved -> log_eval_on_perturbatedMod

## 3b. Deletion (per-spike) — delay

Same experiment on the **delay** model: evaluate the delay checkpoint trained at
`CHECKPOINT_PD_DELETION` across the full deletion sweep.

In [4]:
run_eval(
    deletion_mod, deletion_mod.DeletionSHDNetwork,
    perturbation="deletion", delay_tag="delay",
    ckpt_token=_pd_token,
    eval_levels=deletion_mod.PD_VALUES,
    key_fn=lambda lvl: str(float(lvl)),
)

[deletion/delay] dataset=whole | checkpoint=deletion_whole_delay_pd02.pt
      eval@0.0: 0.8611 +/- 0.0000
      eval@0.2: 0.8383 +/- 0.0009
      eval@0.4: 0.7818 +/- 0.0016
      eval@0.6: 0.6132 +/- 0.0014
      eval@0.8: 0.3382 +/- 0.0109
  saved -> log_eval_on_perturbatedModel\deletion_1stLayer_whole_delay_evalon_pd02.json
[deletion/delay] dataset=part | checkpoint=deletion_part_delay_pd02.pt
      eval@0.0: 0.7375 +/- 0.0000
      eval@0.2: 0.7110 +/- 0.0120
      eval@0.4: 0.6329 +/- 0.0066
      eval@0.6: 0.4681 +/- 0.0100
      eval@0.8: 0.1876 +/- 0.0042
  saved -> log_eval_on_perturbatedModel\deletion_1stLayer_part_delay_evalon_pd02.json
[deletion/delay] dataset=norm | checkpoint=deletion_norm_delay_pd02.pt
      eval@0.0: 0.4139 +/- 0.0000
      eval@0.2: 0.4025 +/- 0.0012
      eval@0.4: 0.3553 +/- 0.0035
      eval@0.6: 0.2658 +/- 0.0057
      eval@0.8: 0.1254 +/- 0.0072
  saved -> log_eval_on_perturbatedModel\deletion_1stLayer_norm_delay_evalon_pd02.json
